# Tutorial: Ejercicio 3.9 en Python - planeacion de puertas for dummies

Audiencia:
- Personas que quieren entender un problema de planeacion semana a semana.
- Personas que prefieren ver primero una solucion en Python antes de AMPL.

Objetivos:
- Entender que decide el modelo.
- Ver los dos balances clave: puertas y operarios.
- Encontrar el plan optimo con una busqueda exacta y explicable.


## Enunciado del ejercicio

La empresa Madera Verde debe cumplir un contrato de `1200` puertas en `5` semanas, con entregas de `100`, `200`, `300`, `400` y `200` puertas en las semanas `1` a `5`.

Se permite:
- guardar inventario para usarlo despues,
- tener faltantes temporales, excepto al final del horizonte,
- contratar personal novato,
- despedir operarios.

Datos del problema:
- inventario inicial: `10` puertas,
- operarios expertos iniciales: `20`,
- cada experto fabrica `8` puertas por semana,
- costo de faltante: `30` USD por puerta y semana,
- costo de inventario: `10` USD por puerta y semana,
- salario semanal por operario: `100` USD,
- costo de despido: `100` USD,
- un experto puede capacitar `5` novatos en una semana y mientras capacita no produce.

Pregunta:
**¿Como planear la produccion al minimo costo cumpliendo todas las condiciones?**


## Mapa del notebook

1. Ver los datos.
2. Entender las decisiones.
3. Entender los balances.
4. Resolver con programacion dinamica.
5. Leer la solucion.
6. Probar una variante guiada.


In [7]:
from functools import lru_cache

SEMANAS = [1, 2, 3, 4, 5]
DEMANDA = {1: 100, 2: 200, 3: 300, 4: 400, 5: 200}
PUERTAS_POR_EXPERTO = 8
COSTO_FALTANTE = 30
COSTO_INVENTARIO = 10
COSTO_SALARIO = 100
COSTO_DESPIDO = 100
INVENTARIO_INICIAL = 10
EXPERTOS_INICIALES = 20
TOPE_EXPERTOS = 45

{
    "demanda": DEMANDA,
    "puertas_por_experto": PUERTAS_POR_EXPERTO,
    "inventario_inicial": INVENTARIO_INICIAL,
    "expertos_iniciales": EXPERTOS_INICIALES,
}


{'demanda': {1: 100, 2: 200, 3: 300, 4: 400, 5: 200},
 'puertas_por_experto': 8,
 'inventario_inicial': 10,
 'expertos_iniciales': 20}

## Paso 1 - Que decide el modelo

Cada semana el modelo decide cinco cosas:
- cuantos expertos producen,
- cuantos expertos actuan como instructores,
- cuantos novatos se estan capacitando,
- cuantos operarios se despiden,
- y cuanto inventario o faltante queda al final de la semana.

La idea mas importante de este notebook es esta:

**si sabemos cuantas personas producen y cuantas capacitan, el inventario y los faltantes salen solos por balance.**


In [8]:
variables = [
    {"variable": "expertos_prod[t]", "significado": "expertos que fabrican puertas en la semana t"},
    {"variable": "instructores[t]", "significado": "expertos que capacitan novatos en la semana t"},
    {"variable": "novatos[t]", "significado": "novatos en capacitacion en la semana t"},
    {"variable": "despedidos[t]", "significado": "operarios despedidos al final de la semana t"},
    {"variable": "inventario[t]", "significado": "puertas sobrantes al final de la semana t"},
    {"variable": "faltantes[t]", "significado": "puertas pendientes al final de la semana t"},
]

variables


[{'variable': 'expertos_prod[t]',
  'significado': 'expertos que fabrican puertas en la semana t'},
 {'variable': 'instructores[t]',
  'significado': 'expertos que capacitan novatos en la semana t'},
 {'variable': 'novatos[t]',
  'significado': 'novatos en capacitacion en la semana t'},
 {'variable': 'despedidos[t]',
  'significado': 'operarios despedidos al final de la semana t'},
 {'variable': 'inventario[t]',
  'significado': 'puertas sobrantes al final de la semana t'},
 {'variable': 'faltantes[t]',
  'significado': 'puertas pendientes al final de la semana t'}]

## Paso 2 - Los dos balances clave

### Balance de puertas

Lo disponible al inicio y durante la semana:
- inventario pasado,
- mas puertas producidas.

Eso debe alcanzar para cubrir:
- la demanda nueva,
- los faltantes que venian de antes,
- y lo que quede como inventario o faltante para el cierre.

### Balance de operarios

Al inicio de cada semana hay un pool de expertos disponibles.
Ese pool se reparte entre:
- expertos que producen,
- expertos que enseñan.

La semana siguiente el pool cambia porque:
- los novatos de esta semana se vuelven expertos,
- los despidos se hacen efectivos.


In [9]:
def siguiente_estado_puertas(inventario_prev, faltante_prev, expertos_prod, demanda, puertas_por_experto=PUERTAS_POR_EXPERTO):
    neto = inventario_prev + puertas_por_experto * expertos_prod - demanda - faltante_prev
    inventario = max(0, neto)
    faltante = max(0, -neto)
    return inventario, faltante


def siguiente_pool_expertos(pool_actual, instructores, despedidos):
    novatos = 5 * instructores
    return pool_actual + novatos - despedidos


{
    "ejemplo_puertas": siguiente_estado_puertas(10, 0, 18, 100),
    "ejemplo_pool": siguiente_pool_expertos(20, 2, 0),
}


{'ejemplo_puertas': (54, 0), 'ejemplo_pool': 30}

## Paso 3 - Resolver el problema exacto

Aqui usamos programacion dinamica con memoizacion.

El estado minimo que necesitamos guardar es:
- semana actual,
- inventario de la semana anterior,
- faltante de la semana anterior,
- pool de expertos disponibles al inicio de la semana.

Usamos un tope de `45` expertos para esta instancia. Es un limite seguro y suficientemente holgado para este horizonte de `5` semanas, y nos permite mantener la busqueda didactica y exacta.


In [10]:
ESPERADO = {
    "costo": 20700,
    "expertos_prod": {1: 18, 2: 29, 3: 35, 4: 35, 5: 32},
    "instructores": {1: 2, 2: 1, 3: 0, 4: 0, 5: 0},
    "novatos": {1: 10, 2: 5, 3: 0, 4: 0, 5: 0},
    "despedidos": {1: 0, 2: 0, 3: 0, 4: 3, 5: 0},
    "inventario": {1: 54, 2: 86, 3: 66, 4: 0, 5: 2},
    "faltantes": {1: 0, 2: 0, 3: 0, 4: 54, 5: 0},
}


def resolver_puertas(costo_faltante=COSTO_FALTANTE, costo_inventario=COSTO_INVENTARIO, tope_expertos=TOPE_EXPERTOS):
    @lru_cache(maxsize=None)
    def dp(semana, inventario_prev, faltante_prev, pool_expertos):
        if pool_expertos > tope_expertos:
            return None
        if semana == len(SEMANAS) + 1:
            return (0, ()) if faltante_prev == 0 else None

        mejor = None
        for instructores in range(pool_expertos + 1):
            expertos_prod = pool_expertos - instructores
            novatos = 5 * instructores
            inventario, faltante = siguiente_estado_puertas(
                inventario_prev=inventario_prev,
                faltante_prev=faltante_prev,
                expertos_prod=expertos_prod,
                demanda=DEMANDA[semana],
            )

            if semana == SEMANAS[-1] and faltante > 0:
                continue

            for despedidos in range(pool_expertos + 1):
                if semana == SEMANAS[-1] and despedidos != 0:
                    continue

                pool_siguiente = siguiente_pool_expertos(
                    pool_actual=pool_expertos,
                    instructores=instructores,
                    despedidos=despedidos,
                )
                if pool_siguiente > tope_expertos:
                    continue

                costo_semana = (
                    COSTO_SALARIO * (expertos_prod + instructores + novatos)
                    + COSTO_DESPIDO * despedidos
                    + costo_inventario * inventario
                    + costo_faltante * faltante
                )

                cola = dp(semana + 1, inventario, faltante, pool_siguiente)
                if cola is None:
                    continue

                candidato = (
                    costo_semana + cola[0],
                    ((expertos_prod, instructores, novatos, despedidos, inventario, faltante),) + cola[1],
                )
                if mejor is None or candidato[0] < mejor[0]:
                    mejor = candidato
        return mejor

    costo_total, plan = dp(1, INVENTARIO_INICIAL, 0, EXPERTOS_INICIALES)

    detalle = []
    resultado = {
        "costo": costo_total,
        "expertos_prod": {},
        "instructores": {},
        "novatos": {},
        "despedidos": {},
        "inventario": {},
        "faltantes": {},
    }

    for semana, fila in enumerate(plan, start=1):
        expertos_prod, instructores, novatos, despedidos, inventario, faltante = fila
        resultado["expertos_prod"][semana] = expertos_prod
        resultado["instructores"][semana] = instructores
        resultado["novatos"][semana] = novatos
        resultado["despedidos"][semana] = despedidos
        resultado["inventario"][semana] = inventario
        resultado["faltantes"][semana] = faltante
        detalle.append(
            {
                "semana": semana,
                "expertos_prod": expertos_prod,
                "instructores": instructores,
                "novatos": novatos,
                "despedidos": despedidos,
                "inventario": inventario,
                "faltantes": faltante,
            }
        )

    resultado["detalle"] = detalle
    return resultado


resultado_base = resolver_puertas()
assert resultado_base["costo"] == ESPERADO["costo"]
assert resultado_base["expertos_prod"] == ESPERADO["expertos_prod"]
assert resultado_base["instructores"] == ESPERADO["instructores"]
assert resultado_base["novatos"] == ESPERADO["novatos"]
assert resultado_base["despedidos"] == ESPERADO["despedidos"]
assert resultado_base["inventario"] == ESPERADO["inventario"]
assert resultado_base["faltantes"] == ESPERADO["faltantes"]

resultado_base


{'costo': 20700,
 'expertos_prod': {1: 18, 2: 29, 3: 35, 4: 35, 5: 32},
 'instructores': {1: 2, 2: 1, 3: 0, 4: 0, 5: 0},
 'novatos': {1: 10, 2: 5, 3: 0, 4: 0, 5: 0},
 'despedidos': {1: 0, 2: 0, 3: 0, 4: 3, 5: 0},
 'inventario': {1: 54, 2: 86, 3: 66, 4: 0, 5: 2},
 'faltantes': {1: 0, 2: 0, 3: 0, 4: 54, 5: 0},
 'detalle': [{'semana': 1,
   'expertos_prod': 18,
   'instructores': 2,
   'novatos': 10,
   'despedidos': 0,
   'inventario': 54,
   'faltantes': 0},
  {'semana': 2,
   'expertos_prod': 29,
   'instructores': 1,
   'novatos': 5,
   'despedidos': 0,
   'inventario': 86,
   'faltantes': 0},
  {'semana': 3,
   'expertos_prod': 35,
   'instructores': 0,
   'novatos': 0,
   'despedidos': 0,
   'inventario': 66,
   'faltantes': 0},
  {'semana': 4,
   'expertos_prod': 35,
   'instructores': 0,
   'novatos': 0,
   'despedidos': 3,
   'inventario': 0,
   'faltantes': 54},
  {'semana': 5,
   'expertos_prod': 32,
   'instructores': 0,
   'novatos': 0,
   'despedidos': 0,
   'inventario': 2,

## Paso 4 - Leer la solucion como historia

La solucion optima dice algo bastante humano:
- al principio se capacita gente nueva,
- eso permite aumentar la produccion en las semanas centrales,
- en la semana 4 se acepta un faltante temporal,
- al final se completa todo el contrato,
- y se despiden `3` operarios antes de la ultima semana para bajar costo.


In [11]:
resultado_base["detalle"]


[{'semana': 1,
  'expertos_prod': 18,
  'instructores': 2,
  'novatos': 10,
  'despedidos': 0,
  'inventario': 54,
  'faltantes': 0},
 {'semana': 2,
  'expertos_prod': 29,
  'instructores': 1,
  'novatos': 5,
  'despedidos': 0,
  'inventario': 86,
  'faltantes': 0},
 {'semana': 3,
  'expertos_prod': 35,
  'instructores': 0,
  'novatos': 0,
  'despedidos': 0,
  'inventario': 66,
  'faltantes': 0},
 {'semana': 4,
  'expertos_prod': 35,
  'instructores': 0,
  'novatos': 0,
  'despedidos': 3,
  'inventario': 0,
  'faltantes': 54},
 {'semana': 5,
  'expertos_prod': 32,
  'instructores': 0,
  'novatos': 0,
  'despedidos': 0,
  'inventario': 2,
  'faltantes': 0}]

## Ejercicio guiado

Prueba ahora un cambio sencillo:
- sube el costo de faltante de `30` a `40` USD,
- y mira como cambia el plan.

Pregunta para pensar antes de ejecutar:
**si el faltante se vuelve mas caro, conviene entrenar antes y fabricar mas temprano?**


In [12]:
resultado_cf40 = resolver_puertas(costo_faltante=40)

{
    "costo_base": resultado_base["costo"],
    "costo_con_faltante_40": resultado_cf40["costo"],
    "expertos_prod_con_faltante_40": resultado_cf40["expertos_prod"],
    "instructores_con_faltante_40": resultado_cf40["instructores"],
    "faltantes_con_faltante_40": resultado_cf40["faltantes"],
}


{'costo_base': 20700,
 'costo_con_faltante_40': 20860,
 'expertos_prod_con_faltante_40': {1: 17, 2: 35, 3: 35, 4: 35, 5: 27},
 'instructores_con_faltante_40': {1: 3, 2: 0, 3: 0, 4: 0, 5: 0},
 'faltantes_con_faltante_40': {1: 0, 2: 0, 3: 0, 4: 14, 5: 0}}

## Comparacion con el libro

En el libro, este problema reporta un costo optimo de `20700` USD y la tabla final de expertos, novatos, despidos, inventario y faltantes que usamos como referencia.

La siguiente celda deja el chequeo explicito dentro del notebook.


In [13]:
comparacion_libro = {
    "costo_coincide": resultado_base["costo"] == ESPERADO["costo"],
    "expertos_prod_coincide": resultado_base["expertos_prod"] == ESPERADO["expertos_prod"],
    "instructores_coincide": resultado_base["instructores"] == ESPERADO["instructores"],
    "novatos_coincide": resultado_base["novatos"] == ESPERADO["novatos"],
    "despedidos_coincide": resultado_base["despedidos"] == ESPERADO["despedidos"],
    "inventario_coincide": resultado_base["inventario"] == ESPERADO["inventario"],
    "faltantes_coincide": resultado_base["faltantes"] == ESPERADO["faltantes"],
    "conclusion": "La respuesta coincide con la del libro.",
}

comparacion_libro


{'costo_coincide': True,
 'expertos_prod_coincide': True,
 'instructores_coincide': True,
 'novatos_coincide': True,
 'despedidos_coincide': True,
 'inventario_coincide': True,
 'faltantes_coincide': True,
 'conclusion': 'La respuesta coincide con la del libro.'}